In [1]:
import random, math
import statistics as stats
from collections import deque

random.seed(42)

class Environment:
    def __init__(self, noise=0.5):
        self.t = 0
        self.noise = noise

    def next_value(self):
        self.t += 1
        trend = 0.05 * self.t
        season = 2.0 * math.sin(2*math.pi*self.t/12)
        value = trend + season + random.gauss(0, self.noise)
        return value


In [2]:
class PredictiveAgent:
    def __init__(self, name, alpha=None, window=6):
        self.name = name
        self.alpha = 0.2 if alpha is None else alpha
        self.window = window
        self.buffer = deque(maxlen=window)
        self.last_forecast = 0.0
        self.err_hist = deque(maxlen=24)

    def update_model(self, x, collective_hint=None):
        z = x if collective_hint is None else 0.7*x + 0.3*collective_hint
        self.buffer.append(z)
        baseline = stats.mean(self.buffer) if len(self.buffer) > 0 else z
        self.last_forecast = (self.alpha * z) + ((1 - self.alpha) * baseline)

    def predict_next(self):
        return self.last_forecast

    def adapt_alpha(self, abs_err):
        self.err_hist.append(abs_err)
        if len(self.err_hist) >= 6:
            recent = sum(list(self.err_hist)[-6:]) / 6.0
            longterm = sum(self.err_hist) / len(self.err_hist)
            if recent > longterm:
                self.alpha = min(0.6, self.alpha + 0.05)
            else:
                self.alpha = max(0.05, self.alpha - 0.02)


In [3]:
class Collective:
    def __init__(self, agents):
        self.agents = agents
        self.last_obs = None
        self.decisions = []

    def aggregate_forecast(self):
        preds = [a.predict_next() for a in self.agents]
        preds_sorted = sorted(preds)
        k = max(1, len(preds)//10)
        trimmed = preds_sorted[k:-k] if len(preds) > 2*k else preds_sorted
        return sum(trimmed) / len(trimmed)

    def decide(self, next_forecast):
        if self.last_obs is None:
            self.decisions.append("hold")
            return "hold"
        direction = "rise" if next_forecast >= self.last_obs else "fall"
        self.decisions.append(direction)
        return direction


In [4]:
env = Environment(noise=0.8)
agents = [PredictiveAgent(f"A{i+1}", alpha=0.15 + 0.02*i) for i in range(10)]
group = Collective(agents)

observations, forecasts, abs_errors = [], [], []

x = env.next_value()
group.last_obs = x
observations.append(x)

for t in range(60):
    x = env.next_value()
    observations.append(x)

    for a in agents:
        a.update_model(x, collective_hint=None)

    hint = sum(a.predict_next() for a in agents) / len(agents)
    for a in agents:
        a.update_model(x, collective_hint=hint)

    f = group.aggregate_forecast()
    forecasts.append(f)
    decision = group.decide(f)

    err = abs(f - x)
    abs_errors.append(err)
    for a in agents:
        a.adapt_alpha(err)

    group.last_obs = x

avg_abs_err = sum(abs_errors)/len(abs_errors)
direction_acc = sum(
    1 for i in range(1, len(observations))
    if (forecasts[i-1] >= observations[i-1]) == (observations[i] >= observations[i-1])
) / (len(observations)-1)
print(f"Average Absolute Error: {avg_abs_err:.3f}")
print(f"Directional Accuracy: {direction_acc*100:.1f}%")


Average Absolute Error: 0.495
Directional Accuracy: 86.7%


In [5]:
from collections import Counter

counts = Counter(group.decisions)
total = sum(counts.values())
stats_summary = {k: f"{(v/total)*100:.1f}%" for k, v in counts.items()}
print("Decision distribution:", stats_summary)

alpha_snapshot = {a.name: round(a.alpha, 3) for a in agents}
print("Adapted alphas:", alpha_snapshot)


Decision distribution: {'rise': '58.3%', 'fall': '41.7%'}
Adapted alphas: {'A1': 0.6, 'A2': 0.6, 'A3': 0.6, 'A4': 0.6, 'A5': 0.6, 'A6': 0.6, 'A7': 0.6, 'A8': 0.6, 'A9': 0.6, 'A10': 0.6}
